# 모듈 02: 상태(State) 관리

모듈 01에서 StateGraph의 4단계 패턴을 익혔습니다.
이번에는 그래프의 핵심인 상태(State)를 깊이 다룹니다.

## 이 노트북에서 할 것

| 단계 | 내용 |
|------|------|
| 1 | TypedDict로 다양한 필드 타입을 가진 상태 설계 |
| 2 | 노드가 상태를 부분 업데이트하는 메커니즘 이해 |
| 3 | 여러 노드가 하나의 상태를 공유하는 패턴 실습 |
| 4 | Annotated + operator.add로 리스트 필드 누적 |

예상 소요 시간: 약 30분  
전제 조건: 모듈 01 완료 (StateGraph 4단계 패턴)

## 학습 목표

1. `TypedDict`로 다양한 필드 타입(str, int, list, bool)을 가진 상태 스키마를 정의할 수 있다
2. 노드 함수가 변경된 필드만 반환해도 LangGraph가 나머지 상태를 자동 병합(merge)함을 이해한다
3. `Annotated` + `operator.add`로 리스트 필드를 덮어쓰지 않고 누적할 수 있다

## 전체 흐름도

```
┌──────────────────────────────────────────────────┐
│                                                  │
│  [1] TypedDict 상태 설계 (다양한 필드 타입)      │
│       ↓                                          │
│  [2] 노드의 부분 업데이트 + 자동 병합 메커니즘   │
│       ↓                                          │
│  [3] 다중 노드 상태 공유 패턴 실습               │
│       ↓                                          │
│  [4] 리듀서: 덮어쓰기 문제 → Annotated 해결     │
│       ↓                                          │
│  [5] 그래프 구조 시각화                          │
│                                                  │
└──────────────────────────────────────────────────┘
```

---

## 섹션 2: TypedDict 상태 설계

### 왜 TypedDict를 사용하는가

- **타입 안정성**: 필드 이름 오타를 IDE가 잡아줌
- **가독성**: 상태 구조를 한눈에 파악 가능
- **문서화**: 상태 설계도 역할

다양한 필드 타입 예시:

```python
class State(TypedDict):
    name: str       # 문자열
    count: int      # 정수
    items: list     # 리스트
    done: bool      # 불리언
```

**중요**: TypedDict는 일반 딕셔너리와 완전히 호환됩니다. 런타임에서는 그냥 dict입니다.

In [ ]:
import os
import operator
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END

notebook_dir = os.path.dirname(os.path.abspath('__file__'))
langgraph_root = os.path.dirname(notebook_dir)
project_root = os.path.dirname(langgraph_root)
langchain_env = os.path.join(project_root, 'LangChain', '.env')
langgraph_env = os.path.join(langgraph_root, '.env')

if os.path.exists(langchain_env):
    load_dotenv(langchain_env)
    print('[OK] LangChain/.env 로드 완료')
elif os.path.exists(langgraph_env):
    load_dotenv(langgraph_env)
    print('[OK] LangGraph/.env 로드 완료')
else:
    print('[FAIL] .env 파일을 찾을 수 없습니다')

api_key = os.getenv('GOOGLE_API_KEY')
print(f'[OK] GOOGLE_API_KEY: {"설정됨" if api_key else "[FAIL] 미설정"}')

In [ ]:
# 다양한 필드 타입을 가진 상태 정의
class OrderState(TypedDict):
    order_id: str       # 주문 번호
    item_count: int     # 주문 수량
    tags: list          # 태그 목록
    is_paid: bool       # 결제 여부

# 초기 상태
state: OrderState = {
    "order_id": "ORD-001",
    "item_count": 3,
    "tags": ["food", "urgent"],
    "is_paid": False
}

print('상태 필드별 타입 확인:')
for key, value in state.items():
    print(f'  {key}: {value!r}  (타입: {type(value).__name__})')

print()
print('[OK] TypedDict는 런타임에서 일반 dict와 동일합니다')
print(f'isinstance(state, dict): {isinstance(state, dict)}')

---

## 섹션 3: 노드의 상태 읽기/쓰기 규칙

핵심: **노드는 바꾸고 싶은 필드만 반환**합니다.

### 자동 병합 메커니즘

```
노드 실행 전 상태:
  {"order_id": "ORD-001", "item_count": 3, "tags": [...], "is_paid": False}

노드가 반환:
  {"is_paid": True}  ← 변경 필드만

LangGraph 병합 결과:
  {"order_id": "ORD-001", "item_count": 3, "tags": [...], "is_paid": True}
  ↑ 나머지 필드는 자동으로 유지됨
```

| 패턴 | 방법 |
|------|------|
| 상태 읽기 | `state["필드명"]` 또는 `state.get("필드명", 기본값)` |
| 상태 업데이트 | `return {"필드명": 새로운_값}` |
| 여러 필드 동시 업데이트 | `return {"필드A": ..., "필드B": ...}` |

In [ ]:
class ProcessState(TypedDict):
    raw_text: str
    word_count: int
    processed: bool

def count_words(state: ProcessState) -> dict:
    """텍스트의 단어 수를 세는 노드 - processed와 word_count만 반환"""
    count = len(state["raw_text"].split())
    return {"word_count": count, "processed": True}
    # raw_text는 반환하지 않아도 유지됨

# 직접 테스트
test_state: ProcessState = {
    "raw_text": "LangGraph로 상태 관리를 배웁니다",
    "word_count": 0,
    "processed": False
}

print(f'실행 전: {test_state}')
update = count_words(test_state)
print(f'노드 반환값: {update}')
print()
print('LangGraph가 자동으로 병합하면:')
merged = {**test_state, **update}  # 병합 시뮬레이션
print(f'병합 결과: {merged}')
print(f'  raw_text 유지: {"raw_text" in merged}')

### 다중 노드 상태 공유 패턴

여러 노드가 하나의 상태를 순서대로 처리하는 패턴:

```
START → [normalize] → [analyze] → [summarize] → END
           ↓               ↓             ↓
        text 정규화    word_count 계산  summary 생성
           ↕               ↕             ↕
                 공유 상태 (PipelineState)
```

각 노드는 담당 필드만 업데이트하고, 다음 노드는 이전 노드가 채운 필드를 읽을 수 있습니다.

In [ ]:
class PipelineState(TypedDict):
    raw_text: str
    clean_text: str
    word_count: int
    summary: str

def normalize(state: PipelineState) -> dict:
    """텍스트를 소문자로 정규화"""
    return {"clean_text": state["raw_text"].lower().strip()}

def analyze(state: PipelineState) -> dict:
    """정규화된 텍스트의 단어 수 계산"""
    count = len(state["clean_text"].split())
    return {"word_count": count}

def summarize(state: PipelineState) -> dict:
    """분석 결과를 요약 문자열로 생성"""
    text = state["clean_text"][:20] + "..." if len(state["clean_text"]) > 20 else state["clean_text"]
    return {"summary": f'텍스트: "{text}", 단어 수: {state["word_count"]}개'}

# 그래프 조립
pipeline = StateGraph(PipelineState)
pipeline.add_node("normalize", normalize)
pipeline.add_node("analyze", analyze)
pipeline.add_node("summarize", summarize)
pipeline.set_entry_point("normalize")
pipeline.add_edge("normalize", "analyze")
pipeline.add_edge("analyze", "summarize")
pipeline.add_edge("summarize", END)
pipeline_app = pipeline.compile()

# 실행
result = pipeline_app.invoke({
    "raw_text": "LangGraph State Management 학습 중",
    "clean_text": "",
    "word_count": 0,
    "summary": ""
})

print('=== 파이프라인 실행 결과 ===')
for key, value in result.items():
    print(f'  {key}: {value!r}')

---

## 섹션 4: 리듀서(Reducer) - 덮어쓰기 vs 누적

**문제**: 기본 동작은 같은 키를 반환하면 덮어씁니다.

```
[노드A] → return {"items": ["a"]}   →  상태: {"items": ["a"]}
[노드B] → return {"items": ["b"]}   →  상태: {"items": ["b"]}  ← "a"가 사라짐!
```

이전 값을 유지하면서 새 값을 추가하려면 **리듀서(Reducer)** 가 필요합니다.

`Annotated`로 필드에 리듀서 함수를 지정합니다:

```python
from typing import Annotated
import operator

class State(TypedDict):
    # 기본: 덮어쓰기
    last_message: str
    # 리듀서 지정: 누적 추가
    history: Annotated[list, operator.add]
```

In [ ]:
# 리듀서 없이 리스트 필드를 업데이트하면?
class NoReducerState(TypedDict):
    step: int
    log: list  # 리듀서 없음

def step1(state: NoReducerState) -> dict:
    return {"step": 1, "log": ["step1 완료"]}

def step2(state: NoReducerState) -> dict:
    return {"step": 2, "log": ["step2 완료"]}  # 이전 log가 덮어써짐

no_reducer_graph = StateGraph(NoReducerState)
no_reducer_graph.add_node("step1", step1)
no_reducer_graph.add_node("step2", step2)
no_reducer_graph.set_entry_point("step1")
no_reducer_graph.add_edge("step1", "step2")
no_reducer_graph.add_edge("step2", END)
no_reducer_app = no_reducer_graph.compile()

result = no_reducer_app.invoke({"step": 0, "log": []})
print('=== 리듀서 없이 실행 ===')
print(f'log: {result["log"]}')
print('[!] step1 로그가 사라졌습니다 → 덮어쓰기 문제')

### 해결책 - Annotated + operator.add

`Annotated[list, operator.add]`를 사용하면:
- LangGraph가 노드 반환값을 기존 리스트에 **이어붙임(extend)**
- 기존 값을 보존하면서 새 값 추가 가능

```python
Annotated[list, operator.add]
          ↑           ↑
       필드 타입    리듀서 함수
                  (기존값 + 새값)
```

`operator.add`는 파이썬 내장 `+` 연산자와 동일합니다:
- 리스트의 경우: `["a"] + ["b"]` → `["a", "b"]`

In [ ]:
# Annotated로 리듀서 지정
class WithReducerState(TypedDict):
    step: int
    log: Annotated[list, operator.add]  # 누적 모드

def step1_r(state: WithReducerState) -> dict:
    return {"step": 1, "log": ["step1 완료"]}

def step2_r(state: WithReducerState) -> dict:
    return {"step": 2, "log": ["step2 완료"]}

reducer_graph = StateGraph(WithReducerState)
reducer_graph.add_node("step1", step1_r)
reducer_graph.add_node("step2", step2_r)
reducer_graph.set_entry_point("step1")
reducer_graph.add_edge("step1", "step2")
reducer_graph.add_edge("step2", END)
reducer_app = reducer_graph.compile()

result = reducer_app.invoke({"step": 0, "log": []})
print('=== Annotated + operator.add 실행 ===')
print(f'log: {result["log"]}')
print('[OK] step1, step2 로그가 모두 누적됩니다')
print()
print('비교 요약:')
print('  리듀서 없음: log = ["step2 완료"]  (덮어씀)')
print('  operator.add: log = ["step1 완료", "step2 완료"]  (누적)')

---

## 섹션 5: 그래프 구조 시각화

`get_graph().print_ascii()`로 두 그래프의 구조를 확인합니다.  
다음 모듈(조건부 흐름)에서는 노드가 더 많아지므로 시각화가 더욱 유용해집니다.

In [ ]:
print('=== 파이프라인 그래프 구조 ===')
pipeline_app.get_graph().print_ascii()

print()
print('=== 리듀서 그래프 구조 ===')
reducer_app.get_graph().print_ascii()

---

## 마무리

### 핵심 패턴 요약

```python
from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, END

# 1. 상태 정의 (일반 필드 + 리듀서 필드 혼합 가능)
class State(TypedDict):
    name: str                               # 덮어쓰기
    log: Annotated[list, operator.add]      # 누적

# 2. 노드: 변경 필드만 반환
def my_node(state: State) -> dict:
    return {"name": "updated"}  # log는 건드리지 않음 → 자동 유지

# 3. 조립 + 실행 (동일)
graph = StateGraph(State)
graph.add_node("my_node", my_node)
graph.set_entry_point("my_node")
graph.add_edge("my_node", END)
app = graph.compile()
result = app.invoke({"name": "", "log": []})
```

### 핵심 API 레퍼런스

| API | 설명 |
|-----|------|
| `TypedDict` | 상태 스키마 정의 (타입 안정성) |
| `state["필드"]` | 현재 상태 값 읽기 |
| `return {"필드": 값}` | 변경 필드만 반환 (자동 병합) |
| `Annotated[list, operator.add]` | 리스트 누적 리듀서 |
| `Annotated[T, fn]` | 커스텀 리듀서 지정 패턴 |

---

## 다음 모듈 예고 + 자기 점검

### 다음 모듈 03 - 조건부 흐름

- 상태 값에 따라 실행 경로를 바꾸는 조건부 엣지
- 라우터 함수 작성 패턴
- `add_conditional_edges()` API

코드 미리보기:

```python
def router(state: State) -> str:
    if state["score"] >= 60:
        return "pass"
    return "fail"

graph.add_conditional_edges("check", router, {"pass": "approve", "fail": "reject"})
```

### 자기 점검 체크리스트

- [ ] TypedDict 상태에서 다양한 필드 타입을 선언할 수 있나요?
- [ ] 노드가 일부 필드만 반환해도 나머지가 유지되는 이유를 설명할 수 있나요?
- [ ] `Annotated[list, operator.add]`와 일반 `list`의 차이를 아나요?